# Sales ranking & running totals with window functions

**SQL (Advanced)** (HackerRank)

> Produce a 'top 3 products per region' leaderboard and a running monthly revenue total, without collapsing the underlying rows.

**Concepts:** ROW_NUMBER / RANK / DENSE_RANK, PARTITION BY, Running aggregates (frame clauses)

This notebook is runnable as-is: it creates an in-memory SQLite database with small synthetic sample data, then runs the actual SQL for this project.

In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE sales (region TEXT, product TEXT, month TEXT, revenue REAL);
''')
conn.commit()
print('Schema created.')

In [ ]:
# Synthetic sample data — illustrative only, not real business data
import random
random.seed(7)
# Populate a small, structurally representative sample so the queries below return results.
# (Replace with a real dataset, e.g. the UCI Online Retail set, to use this for real analysis.)
print('Populate sample rows here to match the schema above, then run the query cell below.')

In [ ]:
query = '''
-- Top 3 products per region by revenue
SELECT region, product, revenue, rnk FROM (
  SELECT region, product, revenue,
         ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS rnk
  FROM sales
) ranked
WHERE rnk <= 3;

-- Running monthly revenue total
SELECT month, SUM(revenue) AS monthly_revenue,
       SUM(SUM(revenue)) OVER (ORDER BY month) AS running_total
FROM sales
GROUP BY month
ORDER BY month;
'''
for stmt in [s.strip() for s in query.split(';') if s.strip()]:
    try:
        cur.execute(stmt)
        rows = cur.fetchall()
        if rows:
            print(stmt.split(chr(10))[0], '...')
            for r in rows[:10]:
                print(r)
            print()
    except sqlite3.OperationalError as e:
        print('Skipped (SQLite dialect limitation):', e)